# 2.5 Your turn: your own chat

Everything through [02.4](02.4-counting-and-averages.ipynb) ran on showcase data, chosen so
the pattern was certain to be there. This notebook has one worked example, to prove the pipes
are connected, and then it is your data and your question.

Your own chat has far fewer people than the IRC corpus does, which makes 02.4's trap sharper
rather than softer — a channel of five people has no room for "the eight regulars" to hide
in, and every person you drop from a comparison moves the answer more, not less.

In [ ]:
from goad_toolkit.datatransforms import CountValues, GroupAgg, Pipeline
from goad_toolkit.visualizer import PlotSettings

from scripts.plots import BarPlot
from wa_analyzer.data import load_own_chat

## 2.5.1 Loading your data

`load_own_chat()` raises when there is no export yet — on purpose. Everything below is about
*your* data; without it there is nothing here to do, and a notebook that quietly runs on
nothing teaches nothing. So if the next cell errors, that is not the notebook breaking — it
is the notebook telling you to go back to [01.3](../lesson1/01.3-your-own-chat.ipynb), run
the preprocessor, and point `current` in `config.toml` at the file it writes.

In [ ]:
own = load_own_chat()
own["length"] = own.message.str.len()
own.head()

## 2.5.2 What's already loaded

`BarPlot` — the class 02.2 derived — is imported above from `scripts/plots.py`, ready to use.
`BarPlotWithError`, 02.4's error-bar version, lives in the same file if the comparison you
pick needs one (`from scripts.plots import BarPlotWithError`). One worked example first, so
you know the pipes connect before you start changing things:

In [ ]:
counts = Pipeline().add(CountValues, column="author").apply(own)

base_settings = PlotSettings(
    figsize=(8, 4),
    title="Messages per person",
    xlabel="messages",
    ylabel="",
)
BarPlot(base_settings).plot(data=counts, x="n", y="author", color="#cccccc")

## 2.5.3 Brainstorming with `goad`

This repo has the `goad` MCP server connected — the same one behind
[CLAUDE.md](../../CLAUDE.md)'s coaching policy. Three tools worth knowing about here, all on
the assistant side of this notebook rather than in it:

- **`goad_analysis_checklist`** walks the six-stage method — the question, the data, the
  shape, the encoding, the critique, the verification — one stage at a time, and will not
  move to the next until you've answered the current one in your own words. Ask your
  assistant to run it for a comparison you want to make in your own chat, and expect to be
  interviewed rather than handed a finding.
- **`goad_search`** / **`goad_list_concepts`** / **`goad_get_concept`** look up a concept
  directly — "which chart family fits a question about who talks to whom", say, or the
  Simpson's-paradox chapter again if 02.3 is worth a second read against your own data.
- **`goad_critique_visual`** runs the four-part visual-critique checklist once you have a
  chart to defend — the same "grey first, is the finding written on the plot, what could you
  delete" questions 02.2 and 02.3 used, put to *your* chart instead of the showcase ones.

The MCP is the coaching layer over `goad_toolkit`, but the toolkit is also its own repo:
[github.com/raoulg/goad_toolkit](https://github.com/raoulg/goad_toolkit). The `docs/`
folder there holds the same ten chapters the MCP serves — pipelines, plot composition, the
five families, visual critique — readable straight from GitHub, with runnable examples in
the source. When you want to browse rather than be interviewed, that is the place.

> **Your turn.**
>
> 1. Make one bar chart that answers a question you actually have about your chat. Order it,
>    grey it, and colour the one bar your title is about — 02.2's guidelines, applied to data
>    nobody chose for you.
> 2. Pick a comparison between two people or two groups. Compute it counting messages, then
>    counting people, the way 02.4 did. Do they agree?
> 3. Before you trust either chart: how many *people* does the claim actually rest on? If the
>    honest answer is two or three, that is worth knowing before the chart is.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from wa_analyzer.data import ROOT, load_own_chat

# 1. Data inladen en filteren (zuivere inhoudelijke berichten)
df = load_own_chat()
clean = df[~df['has_only_media'] & ~df['is_deleted'] & ~df['only_emoji']].copy()
clean['ts'] = pd.to_datetime(clean['timestamp'])

# Bereken context-statistieken voor de verantwoording (Schwabish)
n_total = len(df)
n_clean = len(clean)
n_filtered = n_total - n_clean
n_days = clean['ts'].dt.date.nunique()
min_year = clean['ts'].dt.year.min()
max_year = clean['ts'].dt.year.max()

# Tellen per dagdeel (geaggregeerd)
dagdeel_counts = (
    clean.groupby('day_part', observed=True).size().reset_index(name='n')
)

day_order = ['Ochtend', 'Middag', 'Avond', 'Nacht']
dagdeel_counts['day_part'] = pd.Categorical(
    dagdeel_counts['day_part'], categories=day_order, ordered=True
)
dagdeel_counts = dagdeel_counts.sort_values('day_part').reset_index(drop=True)

# 2. Figuur opzetten (rustige 4-staven grafiek)
fig, ax = plt.subplots(figsize=(8, 5), dpi=150)

# Schwabish Grey First: 3 grijs, exact 1 fel WhatsApp-groen
colors = ['#c8c8c8', '#25D366', '#c8c8c8', '#c8c8c8']
bars = ax.bar(
    dagdeel_counts['day_part'],
    dagdeel_counts['n'],
    color=colors,
    width=0.55,
)

# 3. Clutter verminderen
sns.despine(ax=ax, top=True, right=True, left=False, bottom=False)
ax.set_ylabel('Aantal berichten', fontsize=11, color='#444444', labelpad=10)
ax.tick_params(axis='x', labelsize=12, length=0)
ax.tick_params(axis='y', labelsize=10.5, colors='#555555')

# Y-as in rustige sprongen van 1.000 met Nederlandse punten
ax.set_ylim(0, 3400)
ax.set_yticks([0, 1000, 2000, 3000])
ax.set_yticklabels(['0', '1.000', '2.000', '3.000'])
ax.grid(axis='y', linestyle=':', alpha=0.35, color='#cccccc')

# 4. Direct data labeling (IN de staven bovenaan, behalve Nacht)
whatsapp_dark_green = '#128C7E'

for bar, day, n in zip(
    bars, dagdeel_counts['day_part'], dagdeel_counts['n']
):
    formatted_n = f'{n:,}'.replace(',', '.')
    is_middag = day == 'Middag'

    if day == 'Nacht':
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            n + 40,
            formatted_n,
            ha='center',
            va='bottom',
            fontsize=11,
            color='#666666',
        )
    else:
        text_color = '#ffffff' if is_middag else '#333333'
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            n - 180,
            formatted_n,
            ha='center',
            va='top',
            fontsize=12,
            fontweight='bold',
            color=text_color,
        )

# 5. Annotatie bij de middagpiek (inclusief subtiele uitsplitsing)
middag_bar = bars[1]
ax.annotate(
    'Communicatiepiek: 47% van alle berichten\n(Partner 1.303 | Student 1.535)',
    xy=(middag_bar.get_x() + middag_bar.get_width() / 2, 2838),
    xytext=(middag_bar.get_x() + middag_bar.get_width() / 2 + 0.45, 3100),
    arrowprops=dict(
        arrowstyle='->',
        color=whatsapp_dark_green,
        lw=1.5,
        connectionstyle='arc3,rad=-0.15',
    ),
    fontsize=10.5,
    fontweight='bold',
    color=whatsapp_dark_green,
    ha='left',
    va='center',
)

# 6. Korte, krachtige titel
ax.text(
    0,
    1.08,
    'Communicatiepiek in de middag!',
    transform=ax.transAxes,
    fontsize=15,
    fontweight='bold',
    color='#111111',
)

# 7. Subtiele noot volgens Schwabish-richtlijn (noemer & dataverantwoording)
ax.text(
    0,
    -0.16,
    f'Noot: Gebaseerd op {n_clean:,} inhoudelijke berichten ({n_filtered} media-, verwijderde en pure emoji-berichten gefilterd) over {n_days} actieve dagen ({min_year}–{max_year}).'.replace(',', '.'),
    transform=ax.transAxes,
    fontsize=8,
    style='italic',
    color='#777777',
    ha='left',
)

plt.tight_layout()

# 8. Opslaan als definitieve visualisatie
output_path = ROOT / 'img' / 'dagritme_vergelijking.png'
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_path, bbox_inches='tight')

plt.show()


## 2.5.4 Reflect

- Did counting people change the answer from counting messages? If it did, were you
  surprised, or had you already half-expected it?
- Is there a variable — active period, group role, which sub-conversation someone is in —
  that might split your comparison the way department split Berkeley's admissions in 02.3?
  You don't have to check it now. Just: does one come to mind?
- Look at the chart you made in 2.5.3. If a stranger saw only the chart, thirty seconds, no
  narration from you — would they land on the finding you actually have? If you're not sure,
  `goad_critique_visual` exists for exactly this question, and it's a fair thing to ask your
  assistant to run right now.
- What would it take to change your mind about the comparison you just made? If nothing
  would, it might not be a finding yet.

---

**Where this goes next.** Lesson 3 takes the same comparisons and puts time on the x-axis,
where a new failure appears: the difference between a pattern and the way you smoothed it.